# Сложные условия фильтров
Технология шаблона отчета помогает настроить сложные фильтры за счет передачи списка значений фильтра и списка списков для фильтров useType и date. 

Распределение по подфильтрам происходит так:

- если передан список значений (список списков значений), то в первый подфильтр помещаются все первые элементы списков, во второй - вторые элементы и т.д.
- если один из переданных фильтров задан просто одним значением, то этот фильтр копируется во все подфильтры
- baseGeoFilter и baseDemoFilter никогда не переносятся в подфильтры

Фильтры **base_geo_filter** и **base_demo_filter** определяют генеральную совокупность для всего отчета. Фильтры **geo_filter** и **demo_filter** задают условия для целевой аудитории каждого подфильтра. 

Обратите внимание: фильтры **base_geo_filter** и **base_demo_filter** должны содержать условия для более широкой аудитории, чем **geo_filter** и **demo_filter**. 
*Например, если base_geo_filter - «Россия 100k+», а geo_filter - «Большая Москва», то генеральная совокупность отчета будет сформирована по аудитории «Россия 100k+», а для соответствующего подфильтра дополнительно будет применено ограничение «Большая Москва».*

Пример отчета по расчету аудитории интернет-ресурсов с разными уловиями фильтров (Сross Web). В одном отчете можно использовать более 2 списка списков. 

## Описание задачи

Посчитаем аудиторию нескольких интернет-ресурсам отдельно с разными условиями:
- данные по каждому ресурсу отдельно - Вконтакте (Web Desktop) и Одноклассники (WebMobile);
- данные по возрастным группам и гео по разным условиям - Россия 700 000+, 12+ лет и Россия 700 000-, 12-34 лет;
- данные по каждому ресурсу с разными условиями useType, демо и гео.

Период: 1 - 2 марта 2025 года

Статистики:
- Reach (reach)
- Reach% (reachPer)

## Инициализация

При построении отчета первый шаг в любом ноутбуке - загрузка библиотек, которые помогут обращаться к API Cross Web и работать с данными.

Выполните следующую ячейку, для этого перейдите в нее и нажмите Ctrl+Enter

In [ ]:
import pandas as pd
import json
from mediascope_api.crossweb import tasks as cwt

# Cоздаем объекты для работы с API Cross Web
mtask = cwt.CrossWebTask(check_version=False)
cats = mtask.cats

In [ ]:
# вывод возможных статистик для отчета
cats.get_media_unit().get('statistics')

## Обязательно задание
Объединяем подфильтры в отдельные таски по порядку 1,2,3 и т.д. Это позволяет различить результаты по названиям. 

In [ ]:
# вспомогательные функции для получения расчетов по каждому фильтру в отдельной строке
# реализовано через отправку нескольких заданий

def calc_one_task(task_name=None, task_type=None, 
                  date_filter=None, usetype_filter=None, 
                  geo_filter=None, base_geo_filter=None, 
                  demo_filter=None, base_demo_filter=None, 
                  mart_filter=None, hour_filter=None,
                  slices=None, statistics=None):
    
    task_json = mtask.build_task(task_type=task_type, task_name=task_name, 
                             date_filter=date_filter, usetype_filter=usetype_filter, 
                             geo_filter=geo_filter, base_geo_filter=base_geo_filter, 
                             demo_filter=demo_filter, base_demo_filter=base_demo_filter, 
                             mart_filter=mart_filter, hour_filter=hour_filter,
                             slices=slices, statistics=statistics)

    # Отправляем задание на расчет и ждем выполнения
    task = mtask.wait_task(mtask.send_task(task_json))

    # Получаем результат
    df = mtask.result2table(mtask.get_result(task), project_name=task_name)
    
    return df

def calc_all_subfilters(task_type=None, 
                        date_filter=None, usetype_filter=None, 
                        geo_filter=None, base_geo_filter=None, 
                        demo_filter=None, base_demo_filter=None, 
                        mart_filter=None, hour_filter=None,
                        slices=None, statistics=None):
    
    all_results = []
    
    print(f"\n   Начинаем расчет со всеми заданными фильтрами")
    
    task_name = 'Все заданные фильтры'
    
    df_all = calc_one_task(task_type=task_type, task_name=task_name, 
                  date_filter=date_filter, usetype_filter=usetype_filter, 
                  geo_filter=geo_filter, base_geo_filter=base_geo_filter, 
                  demo_filter=demo_filter, base_demo_filter=base_demo_filter, 
                  mart_filter=mart_filter, hour_filter=hour_filter,
                  slices=slices, statistics=statistics)
    
    all_results.append(df_all)  
    print(f"      ✓ Завершен, получено {len(df_all)} строк")
    
    # Определяем максимальную длину среди всех списков-параметров
    params_lists = {
        'usetype': usetype_filter if usetype_filter else [],
        'mart': mart_filter if mart_filter else [],
        'geo': geo_filter if geo_filter else [],
        'demo': demo_filter if demo_filter else [],
        'hour': mart_filter if mart_filter else []
    }
    
    # Находим максимальную длину среди всех списков
    max_length = max([len(lst) for lst in params_lists.values() if lst]) if params_lists else 0
    
    # Выполняем поэлементные расчеты
    for i in range(max_length):
        # Формируем название расчета на основе индекса и значений
        calc_parts = []
        
        # Берем i-й элемент из каждого списка (если он существует)
        current_usetype = None
        if usetype_filter and i < len(usetype_filter):
            current_usetype = [usetype_filter[i]] if not isinstance(usetype_filter[i], list) else usetype_filter[i]
            # Преобразуем в читаемое название
            calc_parts.append(f"useType_{current_usetype}")
        else:
            current_usetype = usetype_filter if usetype_filter else None
        
        current_mart = None
        if mart_filter and i < len(mart_filter):
            current_mart = mart_filter[i]
            # Преобразуем в читаемое название
            calc_parts.append(f"mart_{i+1}_{current_mart}")
        else:
            current_mart = mart_filter if mart_filter else None
        
        current_geo = None
        if geo_filter and i < len(geo_filter):
            current_geo = geo_filter[i]
            calc_parts.append(f"geo_{i+1}_{current_geo}")
        else:
            current_geo = geo_filter
        
        current_demo = None
        if demo_filter and i < len(demo_filter):
            current_demo = demo_filter[i]
            calc_parts.append(f"demo_{i+1}_{current_demo}")
        else:
            current_demo = demo_filter
        
        current_hour = None
        if hour_filter and i < len(hour_filter):
            current_hour = hour_filter[i]
            calc_parts.append(f"hour_{i+1}_{current_hour}")
        else:
            current_hour = hour_filter
        
        # Формируем название расчета
        if calc_parts:
            calc_label = " + ".join(calc_parts)
        else:
            calc_label = f"Расчет_{i+1}"
        
        print(f"\n   Расчет {i+1}/{max_length}: {calc_label}")
        
        task_name_single = f'{calc_label}'
        
        df_single = calc_one_task(task_type=task_type, task_name=task_name_single, 
                  date_filter=date_filter, usetype_filter=current_usetype, 
                  geo_filter=current_geo, base_geo_filter=base_geo_filter, 
                  demo_filter=current_demo, base_demo_filter=base_demo_filter, 
                  mart_filter=current_mart, hour_filter=current_hour,
                  slices=slices, statistics=statistics)
        
        all_results.append(df_single)
        print(f"      ✓ Завершен, получено {len(df_single)} строк")
    
    # Объединяем все результаты
    if all_results:
        final_df = pd.concat(all_results, ignore_index=True)
        return final_df
    else:
        print("\n❌ Не получено ни одного результата")
        return pd.DataFrame()


## Задания

Перейдем к формированию заданий.


### Задание №1. Расчет аудитории Вконтакте (Web Desktop) и Одноклассники (WebMobile)
В данном задании присутствуют два типа подфильтра:

по типам пользования интернетом (usetype_filter):
1. Web desktop (id 1);
2. Web mobile (id 2).

по медиа (mart_filter):
1. Вконтакте (id 1150);
2. ОК (id 1093)

Таким образом у нас появляется два подрасчета с подфильтрами:
1. Вконтакте (Web desktop)
2. ОК (Web mobile)

In [ ]:
date_filter = [('2025-03-01', '2025-03-02')]

# Задаем фильтр по типам пользования интернетом
#usetype_filter = None
usetype_filter = [[1], [2]] # задаем список списков с первым элементом Web desktop и вторым - Web mobile

# Задаем фильтр по географии
geo_filter = None

# Задаем базовый фильтр по географии
base_geo_filter = None

# Задаем фильтр по демографии
demo_filter = None

# Задаем базовый фильтр по демографии
base_demo_filter = None

# Задаем фильтр по медиа (задаем список - первый элемент Вконтакте, второй - ОК)
#mart_filter = None
mart_filter = ['crossMediaResourceId IN (1150)', 'crossMediaResourceId IN (1093)']

# Задаем почасовой фильтр
hour_filter = None 

# Указываем список срезов, чтобы сформировать структуру расчета
slices = ["researchDate"]

# Указываем список статистик для расчета
statistics = [
    'reach', 'reachPer'
    ] 

df_res = calc_all_subfilters(task_type='media', 
                    date_filter=date_filter, usetype_filter=usetype_filter, 
                    geo_filter=geo_filter, base_geo_filter=base_geo_filter, 
                    demo_filter=demo_filter, base_demo_filter=base_demo_filter, 
                    mart_filter=mart_filter, hour_filter=hour_filter,
                    slices=slices, statistics=statistics)
df_res

### Сохраняем в Excel

In [ ]:
with pd.ExcelWriter(mtask.get_excel_filename('subfilters')) as writer:
    df_res.to_excel(writer, sheet_name='Report', index=False)

### Задание №2. Расчет аудитории по возрастным группам и гео по разным условиям - Россия 700 000+, 12+ лет и Россия 700 000-, 12-34 лет
В данном задании присутствуют два типа подфильтра:

по географии (geo_filter):
1. города 700К+ (cityPop = 3);
2. города 700К- (cityPop in (4,5,6).

по демографии (demo_filter):
1. возраст 12+ (age >= 12);
2. возраст 12-34 (age >= 12 and age <= 34).

Таким образом у нас появляется два подрасчета с подфильтрами:
1. города 700К+, возраст 12+;
2. города 700К-, возраст 12-34;

In [ ]:
date_filter = [('2025-03-01', '2025-03-02')]

# Задаем фильтр по типам пользования интернетом
usetype_filter = None

# Задаем фильтр по географии
geo_filter = ['cityPop = 3', 'cityPop in (4, 5, 6)'] # первый элемент списка - 700000+, второй - 700000-

# Задаем базовый фильтр по географии
base_geo_filter = None

# Задаем фильтр по демографии
demo_filter = ['age >= 12', 'age >= 12 and age <= 34'] # первый элемент списка - 12+, второй - 12-34

# Задаем базовый фильтр по демографии
base_demo_filter = None

# Задаем фильтр по медиа
mart_filter = None

# Задаем почасовой фильтр
hour_filter = None 

# Указываем список срезов, чтобы сформировать структуру расчета
slices = ["researchDate"]

# Указываем список статистик для расчета
statistics = [
    'reach', 'reachPer'
    ] 

df_res = calc_all_subfilters(task_type='media', 
                    date_filter=date_filter, usetype_filter=usetype_filter, 
                    geo_filter=geo_filter, base_geo_filter=base_geo_filter, 
                    demo_filter=demo_filter, base_demo_filter=base_demo_filter, 
                    mart_filter=mart_filter, hour_filter=hour_filter,
                    slices=slices, statistics=statistics)
df_res

### Сохраняем в Excel

In [ ]:
with pd.ExcelWriter(mtask.get_excel_filename('subfilters')) as writer:
    df_res.to_excel(writer, sheet_name='Report', index=False)

### Задание №3. Расчет аудитории по каждому ресурсу с разными условиями useType, демо и гео.
В данном задании присутствуют четыре типа подфильтра:

по типам пользования интернетом (usetype_filter):
1. Web desktop (id 1);
2. Web mobile (id 2).

по географии (geo_filter):
1. города 700К+ (cityPop = 3);
2. города 700К- (cityPop in (4,5,6).

по демографии (demo_filter):
1. возраст 12+ (age >= 12);
2. возраст 12-34 (age >= 12 and age <= 34).

по медиа (mart_filter):
1. Вконтакте (id 1150);
2. ОК (id 1093)

Таким образом у нас появляется два подрасчета с подфильтрами:
1. Вконтакте (Web desktop), города 700К+, возраст 12+;
2. ОК (Web mobile), города 700К-, возраст 12-34.

In [ ]:
date_filter = [('2025-03-01', '2025-03-02')]

# Задаем фильтр по типам пользования интернетом
#usetype_filter = None
usetype_filter = [[1], [2]] # задаем список списков с первым элементом Web desktop и вторым - Web mobile

# Задаем фильтр по географии
#geo_filter = None 
geo_filter = ['cityPop = 3', 'cityPop in (4, 5, 6)'] # первый элемент списка - 700000+, второй - 700000-

# Задаем базовый фильтр по географии
base_geo_filter = None

# Задаем фильтр по демографии
#demo_filter = None
demo_filter = ['age >= 12', 'age >= 12 and age <= 34'] # первый элемент списка - 12+, второй - 12-34

# Задаем базовый фильтр по демографии
base_demo_filter = None

# Задаем фильтр по медиа
#mart_filter = None
mart_filter = ['crossMediaResourceId IN (1150)', 'crossMediaResourceId IN (1093)']

# Задаем почасовой фильтр
hour_filter = None 

# Указываем список срезов, чтобы сформировать структуру расчета
slices = ["researchDate"]

# Указываем список статистик для расчета
statistics = [
    'reach', 'reachPer'
    ] 

df_res = calc_all_subfilters(task_type='media', 
                    date_filter=date_filter, usetype_filter=usetype_filter, 
                    geo_filter=geo_filter, base_geo_filter=base_geo_filter, 
                    demo_filter=demo_filter, base_demo_filter=base_demo_filter, 
                    mart_filter=mart_filter, hour_filter=hour_filter,
                    slices=slices, statistics=statistics)
df_res

### Сохраняем в Excel

In [ ]:
with pd.ExcelWriter(mtask.get_excel_filename('subfilters')) as writer:
    df_res.to_excel(writer, sheet_name='Report', index=False)